# Model V2.5 - 15-Minute Price Prediction

In [ ]:
import pandas as pd

In [ ]:
# load the 15-minute engineered feature CSV (produced by V2.5_feature_engineering.ipynb)
df = pd.read_csv('../data/convertData/V2.5_15min_features.csv')
df

### Create train & test data set

In [ ]:
# drop the target variable and the datetime column from the feature set
X = df.drop(columns = ['price', 'datetime'])

# set the dependent variable (what we want to predict = price)
y = df['price']

In [ ]:
from sklearn.model_selection import train_test_split

# spliting the data into training and testing sets (80/20, no shuffle - time series!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, shuffle=False)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
# print the list of feature column names (already clean from V2.5_feature_engineering.ipynb)
print('Feature columns used in V2.5:', X_train.columns.tolist())

### TRAIN V2.5 XGBOOST REGRESSION MODEL

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

In [ ]:
# instantiate the model with the same hyperparameters as V2.5
model_v25 = XGBRegressor(
    objective='reg:squarederror',
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    random_state=42)

# train the model on the training data
model_v25.fit(X_train, y_train)

In [ ]:
# test the model on the test set and get predictions
y_pred = model_v25.predict(X_test)

# k is the number of features in the test set
k = X_test.shape[1]
# n is the number of samples in the test set
n = len(X_test)

# calculate mse, rmse, mae, r2 score and adjusted r2 score
mse_score = mean_squared_error(y_true=y_test, y_pred=y_pred)

rmse_score = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_pred))

mae_score = mean_absolute_error(y_true=y_test, y_pred=y_pred)

r2_score_val = r2_score(y_true=y_test, y_pred=y_pred)

adjusted_r2 = 1 - (1 - r2_score_val) * (n - 1) / (n - k - 1)

# print the evaluation results
print('================ V2.5 XGBoost Evaluation ================')
print('Mean Absolute Error (MAE)            :', mae_score)
print('Mean Squared Error (MSE)             :', mse_score)
print('Root Mean Squared Error (RMSE)       :', rmse_score)
print('R2 Score (R2)                        :', r2_score_val)
print('Adjusted R2                           :', adjusted_r2)
print('==========================================================')

# Model V2.5 Performance Report — 15-Minute Engineered XGBoost

## 1. Executive Summary

Model V2.5 upgrades from hourly (V1/V2) to 15-minute resolution with the same
full engineered feature set (lag, rolling, calendar, holiday, weather proxies).
The 80/20 chronological split and XGBoost hyperparameters are identical to V1/V2.
The only differences: 4x more rows (15-min data) and finer-grained lag/rolling
windows tuned for 15-minute steps.

## 2. Quantitative Evaluation — Full 4-Model Comparison

| Metric      | V1 (h, weather only) | V1.5 (15-min, weather only) | V2 (h, engineered) | **V2.5 (15-min, engineered)** |
|-------------|----------------------|-----------------------------|--------------------|-------------------------------|
| MAE         | 33.13               | 32.19                       | 7.22               | **2.82**                      |
| MSE         | 2,147.37            | 2,096.21                    | 213.81             | **67.55**                     |
| RMSE        | 46.34               | 45.78                       | 14.62              | **8.22**                      |
| R² Score    | 0.107               | 0.125                       | 0.911              | **0.972**                     |
| Adjusted R² | 0.106               | 0.125                       | 0.910              | **0.972**                     |

**Key takeaways:**
- **V1 → V1.5:** Weather-only models barely improve with finer resolution (R² 0.107 → 0.125).
- **V1 → V2:** Feature engineering is the real game-changer (R² 0.107 → 0.911).
- **V2 → V2.5:** 15-minute resolution + engineered features gives the best result (R² 0.911 → 0.972).

## 3. Analytical Interpretation

**V2.5 is the best model so far.** R² of 0.972 means the model explains 97.2 %
of price variance — extremely strong for a tabular model with basic features.
The average error (MAE = 2.82 EUR/MWh) is well within practical trading tolerance.

**Why V2.5 beats V2 (and why V1.5 barely improves over V1):**
- **Feature engineering matters most.** V1 → V2 added engineered features (same hourly resolution) and R² jumped from 0.107 to 0.911. V1 → V1.5 just increased resolution without adding features, and R² barely moved (0.107 → 0.125). This proves that time-series features (lags, rolling, calendar) are the primary driver of accuracy.
- **4x more training data than V2.** V2 had ~21,000 hourly rows; V2.5 has ~84,000 training rows (80 % of 105,216). More samples → better generalization.
- **Finer temporal resolution.** The model sees intra-hour price dynamics (every 15 minutes instead of every hour), learning short-term patterns that hourly aggregation smoothed away.
- **More lag granularity.** Lags at 1, 2, 4, 8, 16, 32, 96, 672 steps let the model capture 15-min/30-min/1h/2h dependencies that hourly lags could not.

**Error analysis:**
- RMSE (8.22) is low but still ~3x MAE (2.82), suggesting some outlier hours
  with large prediction errors — likely during extreme price spikes.
- This gap (8.22 vs 2.82) is much smaller than V2's (14.62 vs 7.22), indicating
  V2.5 handles volatile periods better.

## 4. Which Model Should You Use?

| Use case                      | Recommended model |
|-------------------------------|-------------------|
| Quick baseline / demo         | V1 (RMSE 46.34)  |
| Baseline at 15-min resolution | V1.5 (RMSE 45.78)|
| Hourly forecasting            | V2 (RMSE 14.62)  |
| **15-min / intraday trading** | **V2.5 (RMSE 8.22)** |

## 5. Next Steps

V2.5 is demonstrably strong but can still be improved:

- **Hyperparameter tuning** — V2.5 uses the same naive params as V1 (n_estimators=100,
  learning_rate=0.1). A tuned model could push R² above 0.98.
- **Time-series cross-validation** — verify stability across different
  seasonal windows.
- **Prediction script** — write `predict_v2.5.py` for live 15-min forecasts.
- **Feature importance analysis** — identify which 15-min features drive predictions
  most (likely `price_lag_672` and `price_rolling_mean_7d`).